# Reproducing *Robust Lightweight Deep Learning Models for Oral Cancer Screening* on public data

This notebook runs the whole re-implementation end to end: download the open datasets, build the manifests,
precompute SAM masks and image-quality scores, train the experiment grid (paper Tables IV–VIII plus the held-out
source), run the noise stress test (Table IX) and interpretability, and rebuild every table and figure of the report.

**How to use it**
- Run it from the `oral-screen/` folder (the folder that contains `src/`), in the project's environment
  (`pip install -r requirements.txt`, plus a CUDA build of PyTorch).
- Set `MODE` below. `"smoke"` checks the pipeline in minutes (1 seed, 2 epochs, a few configurations);
  `"full"` reproduces the report (5 seeds × 50 epochs; roughly 15–20 GPU-hours on 2× T4).
- Every training step skips configurations that already have a `summary.json`, so the notebook can be stopped and
  re-run, and results downloaded from Kaggle can be dropped into `results/` first.
- Smoke-mode results go to a separate `results_smoke/` copy so they never mix with the real ones.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path

MODE = "smoke"          # "smoke" or "full"
GPUS = 1                # set 2 on a 2-GPU machine (e.g. Kaggle T4 x2) to run two grid shards in parallel
WORKERS = 0             # DataLoader workers; 0 avoids the deadlock seen on Kaggle's 4 vCPUs

ROOT = Path.cwd()
assert (ROOT / "src" / "run_grid.py").exists(), "run this notebook from the oral-screen/ folder"
SEEDS = ["0"] if MODE == "smoke" else ["0", "1", "2", "3", "4"]
EPOCHS = "2" if MODE == "smoke" else "50"
PY = sys.executable

def sh(*args, env=None):
    # Run a project script from the project root and stream its output; stop the notebook on failure.
    cmd = [PY, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, cwd=ROOT, env={**os.environ, **(env or {})})
    if p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Data

Two open sources replace the paper's private ~30,000-image corpus:
- **Mendeley** Oral Images Dataset (doi 10.17632/mhjyrn35p4.2), `original_data` only: 165 benign → non-suspicious,
  158 malignant → suspicious.
- **Kaggle** `shivam17299/oral-cancer-lips-and-tongue-images`: `cancer` → suspicious, `non-cancer` → non-suspicious.
  Downloading it may need `~/.kaggle/kaggle.json` (or `KAGGLE_USERNAME` / `KAGGLE_KEY`).

In [ ]:
if not any((ROOT / "data/raw").glob("*/*")):
    sh("src/download_data.py", "--out", "data/raw")
else:
    print("data/raw already populated; skipping download")

`prepare_data.py` maps folder labels to the binary schema, drops unreadable and blurry images, removes near-duplicates
(flip/rotation-invariant perceptual hash), and makes a stratified 60/20/20 split. Three manifests are built:
- `manifest.csv` — the natural split (also the input for SAM masks and IQA scores);
- `manifest_train5.csv` — **the main manifest**: training split subsampled to the paper's 5:1 imbalance, val/test keep
  every suspicious image;
- `manifest_external.csv` — the Kaggle source held out entirely as an "external" test split.

In [ ]:
common = ["--raw", "data/raw", "--out", "data"]
if not (ROOT / "data/manifest.csv").exists():
    sh("src/prepare_data.py", *common)
if not (ROOT / "data/manifest_train5.csv").exists():
    sh("src/prepare_data.py", *common, "--neg-pos-ratio", "5", "--rebalance-scope", "train",
       "--name", "manifest_train5.csv")
if not (ROOT / "data/manifest_external.csv").exists():
    sh("src/prepare_data.py", *common, "--neg-pos-ratio", "5", "--rebalance-scope", "train",
       "--external-source", "kaggle_lips_tongue", "--name", "manifest_external.csv")

import pandas as pd
for name in ["manifest.csv", "manifest_train5.csv", "manifest_external.csv"]:
    m = pd.read_csv(ROOT / "data" / name)
    print(f"\n{name}"); print(pd.crosstab(m["split"], m["label"], margins=True))

### Offline inputs for Tables IV and V
- **SAM masks** (Table IV, "SAM output" augmentation) need the SAM ViT-B checkpoint
  (`checkpoints/sam_vit_b_01ec64.pth`, ~375 MB, from Meta's official `segment-anything` release).
- **NIQE and CLIP-IQA scores** (Table V) are computed with `pyiqa`.

In [ ]:
ckpt = ROOT / "checkpoints/sam_vit_b_01ec64.pth"
if not ckpt.exists():
    import urllib.request
    ckpt.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", ckpt)
if not (ROOT / "data/sam_masks").exists():
    sh("src/sam_masks.py", "--manifest", "data/manifest.csv")
if not (ROOT / "data/iqa_scores.csv").exists():
    sh("src/iqa_scores.py", "--manifest", "data/manifest.csv")

## 2. Training grid

`src/run_grid.py` defines every experiment (see `experiments()` there) and skips configurations that already finished.
Protocol: Adam, lr 1e-3 with ReduceLROnPlateau on validation sensitivity × specificity, 50 epochs, effective batch 32,
checkpoint = best validation sens × spec, results = test-set mean ± std over seeds.

| Group | Paper table | What it runs |
|---|---|---|
| `table4` | IV | 6 students × {none, geometric, zoom, colour jitter, SAM masking}, BCE |
| `table5` | V | 6 students × NIQE / CLIP-IQA loss reweighting |
| `table6` | VI | 6 students × {under-, over-sampling, WCE, focal, Balanced Mix-up α = 0.1/0.2/0.3} |
| `recipe` | — | MobileViTv2 + geometric + WCE, checkpoints kept for Table IX and interpretability |
| `table7` | VII | the recipe, 5-fold CV (train+val pooled) |
| `teachers` | VIII | ViT-B/16, Swin-B (**lr 1e-5** — they collapse at 1e-3 and 1e-4), EfficientNetV2-L @384 px; one seed |
| `table8` | VIII | MobileViTv2 distilled from each teacher, KD weight β = 0.5 / 1.0, T = 4 |
| `external` | — | baseline and recipe on `manifest_external.csv` |

`teachers` must finish before `table8`; the grid keeps them in that order. In smoke mode only a small subset runs.

In [ ]:
GROUPS = ["table4", "table5", "table6", "recipe", "table7", "teachers", "table8", "external"]
args = ["src/run_grid.py", "--groups", *GROUPS, "--seeds", *SEEDS, "--epochs", EPOCHS, "--workers", str(WORKERS)]

# what is still to do
sh(*args, "--list")

In [ ]:
if MODE == "smoke":
    # A handful of single configurations through train.py directly, written to results_smoke/.
    smoke = [
        ["--model", "mobilevitv2"],
        ["--model", "mobilevitv2", "--aug", "geometric", "--loss", "wce", "--keep-ckpt"],
        ["--model", "edgenext_s", "--sampling", "under"],
        ["--model", "mobilevitv2", "--iqa", "clipiqa"],
    ]
    for cfg in smoke:
        name = "_".join(c.lstrip("-") for c in cfg)
        sh("src/train.py", "--manifest", "data/manifest_train5.csv", "--seeds", *SEEDS, "--epochs", EPOCHS,
           "--workers", str(WORKERS), "--batch", "16", "--accum", "2", *cfg, "--out", f"results_smoke/{name}")
elif GPUS > 1:
    # one shard per GPU, run in parallel (this is what the Kaggle kernel does)
    procs = []
    for g in range(GPUS):
        log = open(ROOT / f"grid_shard{g}.log", "w")
        procs.append(subprocess.Popen([PY, *args, "--num-shards", str(GPUS), "--shard", str(g)], cwd=ROOT,
                                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)},
                                      stdout=log, stderr=subprocess.STDOUT))
    print("shards started; follow grid_shard*.log")
    print("exit codes:", [p.wait() for p in procs])
else:
    sh(*args)

## 3. Robustness (Table IX) and interpretability

The kept recipe checkpoints are evaluated on the test set under Gaussian, stripe and salt-and-pepper noise, and
Grad-CAM++ / attention-rollout maps are drawn for true positives and false negatives.

In [ ]:
recipe = "results_smoke/model_mobilevitv2_aug_geometric_loss_wce_keep-ckpt" if MODE == "smoke" \
    else "results/manifest_train5/mobilevitv2_geometric_wce"
if not (ROOT / recipe / "robustness.csv").exists():
    sh("src/robustness.py", "--run-dir", recipe, "--manifest", "data/manifest_train5.csv")
    sh("src/explain.py", "--run-dir", recipe, "--manifest", "data/manifest_train5.csv")
print(pd.read_csv(ROOT / recipe / "robustness.csv", header=[0, 1], index_col=[0, 1]).round(1))

## 4. Tables and figures

`src/report.py` aggregates `results/manifest_train5/` into `results/tables/` (mean ± std over seeds, best-GM seed pair,
GM of the means, AUROC, NPV adjusted to the paper's 16.5% prevalence). `report/make_figures.py` redraws the report
figures. In smoke mode these steps use whatever full results are already in `results/`.

In [ ]:
sh("src/report.py")
sh("report/make_figures.py")

In [ ]:
from IPython.display import Markdown, display
for name in ["table4_augmentation", "table5_iqa", "table6_imbalance", "table7_cv", "table8_distillation",
             "table9_robustness", "external_source_heldin", "external_source"]:
    f = ROOT / "results/tables" / f"{name}.md"
    if f.exists():
        display(Markdown(f"### {name}\n\n" + f.read_text(encoding="utf-8")))

In [ ]:
# The plots are PDFs for LaTeX (open them from report/figures/); the interpretability montages are PNGs.
from IPython.display import Image
print("figures:", ", ".join(f.name for f in sorted((ROOT / "report/figures").iterdir())))
for f in sorted((ROOT / recipe / "explain").glob("*.png")):
    print(f.name); display(Image(filename=str(f), width=900))

## 5. Report

The IEEE-style report is `report/report.tex`. Rebuild it with
`latexmk -pdf report.tex` inside `report/` (needs a LaTeX installation with IEEEtran).

**Deviations from the paper** (all documented in the report): public data (~420 images instead of ~30,000);
`mnasnet_100` instead of NASNet-Mobile; ViT-B/16 and Swin-B teachers at lr 1e-5; EfficientNetV2-L teacher at 384 px;
5 seeds instead of 10; DataLoader workers = 0 on Kaggle.